In [1]:
import os
import re
import time
from pathlib import Path
from datetime import datetime

import joblib
import mlflow
import mlflow.xgboost
import xgboost as xgb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from mlflow.models import infer_signature

In [ ]:
mlflow.set_tracking_uri("http://localhost:5000")

EXPERIMENT_NAME = "xgb_Mnist_Prediction_Experiments"
REGISTERED_MODEL_NAME = "MNIST_CLASSIFICATION_Predictor"

mlflow.set_experiment(EXPERIMENT_NAME)


OUTPUT_DIR = Path("models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("MLflow version:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())



def get_next_experiment_version(experiment_name):
    experiment = mlflow.get_experiment_by_name(experiment_name)

    if experiment is None:
        experiment_id = mlflow.create_experiment(experiment_name)
        return 1

    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["attributes.start_time DESC"]
    )

    if runs.empty:
        return 1

    versions = []

    for run_name in runs.get(
        "tags.mlflow.runName", pd.Series(dtype=str)
    ).dropna():
        match = re.search(r"_v(\d+)_", run_name)
        if match:
            versions.append(int(match.group(1)))

    return max(versions, default=0) + 1


train_csv = Path(r"C:\Users\USER\OneDrive\Desktop\Aya\Machan learning\Machine_learning_Fils\Mnist\mnist_train.csv")

if not train_csv.exists():
    raise FileNotFoundError(
        f"Cannot find {train_csv.resolve()}. "
        "Put the CSV in the notebook's working directory "
        "or update train_csv to its actual path."
    )

df = pd.read_csv(train_csv)

if "label" not in df.columns:
    raise ValueError(
        "The training CSV must contain a 'label' column."
    )

Number_mnist_X = df.drop(columns=["label"]).astype("float32") / 255.0
Number_mnist_y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    Number_mnist_X,
    Number_mnist_y,
    test_size=0.2,
    random_state=42,
    stratify=Number_mnist_y
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_test))
print("Number of features:", X_train.shape[1])



xgb_params = {
    "objective": "multi:softprob",
    "num_class": 10,
    "n_estimators": 4
    00,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "eval_metric": "mlogloss",
    "random_state": 42,
    "n_jobs": -1
}


@mlflow.trace
def predict_Mnist(model, features):
    """Run predictions and record an MLflow trace."""
    return model.predict(features)


version = get_next_experiment_version(EXPERIMENT_NAME)

run_name = (
    f"{EXPERIMENT_NAME}_v{version}_"                        ## xgb_fashion_Prediction_Experiments_v1
    f"{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
)

start_time = time.time()

with mlflow.start_run(run_name=run_name) as run:

    print("Run ID:", run.info.run_id)
    print("Experiment version:", version)

    # Record useful metadata
    mlflow.set_tags({
        "model_family": "XGBoost",
        "task": "multiclass_classification",
        "dataset": "NUMBER-MNIST",
        "experiment_version": str(version)
    })

    # Log dataset location and schema
    mlflow.log_param("dataset_path", str(train_csv.resolve()))
    mlflow.log_param("dataset_rows", len(df))
    mlflow.log_param("number_of_features", Number_mnist_X.shape[1])

    dataset = mlflow.data.from_pandas(
        df,
        source=str(train_csv.resolve()),
        targets="label"
    )

    mlflow.log_input(dataset, context="training")

    # Log model parameters
    mlflow.log_params(xgb_params)

    # Train the model
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(X_train, y_train)

    # Predict and evaluate
    xgb_pred_mnist = predict_Mnist(xgb_model, X_test)

    accuracy = accuracy_score(y_test, xgb_pred_mnist)
    report = classification_report(
        y_test,
        xgb_pred_mnist,
        output_dict=True,
        zero_division=0
    )

    # Log metrics
    mlflow.log_metric("accuracy", float(accuracy))
    mlflow.log_metric("accuracy_percent", float(accuracy * 100))
    mlflow.log_metric(
        "model_size_mb",
        float(
            len(joblib.dumps(xgb_model)) / (1024 * 1024)
        ) if hasattr(joblib, "dumps") else 0.0
    )

    # Log selected classification metrics
    for average in ["macro avg", "weighted avg"]:
        mlflow.log_metric(
            f"f1_{average.replace(' ', '_')}",
            float(report[average]["f1-score"])
        )

    print(f"Validation accuracy: {accuracy * 100:.2f}%")

    pkl_path = OUTPUT_DIR / f"xgb_model_Mnist_{version}.pkl"

    joblib.dump(xgb_model, pkl_path)

    # Log the file directly in Run Artifacts
    mlflow.log_artifact(
        str(pkl_path),
        artifact_path="joblib_model"
    )

    mlflow.log_metric(
        "pkl_size_mb",
        pkl_path.stat().st_size / (1024 * 1024)
    )


    input_example = X_test.iloc[:5].copy()
    example_predictions = xgb_model.predict(input_example)

    signature = infer_signature(
        input_example,
        example_predictions
    )

    model_info = mlflow.xgboost.log_model(
        xgb_model,
        name="xgboost-model",
        registered_model_name=REGISTERED_MODEL_NAME,
        input_example=input_example,
        signature=signature,
        pip_requirements=[
            f"mlflow=={mlflow.__version__}",
            f"xgboost=={xgb.__version__}",
            f"pandas=={pd.__version__}",
            "numpy"
        ]
    )

    print("Logged model URI:", model_info.model_uri)
    print("Logged model ID:", model_info.model_id)


    local_model_dir = (
        OUTPUT_DIR / f"mlflow_xgboost_model_v{version}"
    )

    mlflow.xgboost.save_model(
        path=str(local_model_dir),
        xgb_model=xgb_model,
        input_example=input_example,
        signature=signature,
        pip_requirements=[
            f"mlflow=={mlflow.__version__}",
            f"xgboost=={xgb.__version__}",
            f"pandas=={pd.__version__}",
            "numpy"
        ]
    )


    # python_env.yaml, conda.yaml, and the model file.
    print("Local MLflow model files:")
    for file_path in sorted(local_model_dir.rglob("*")):
        if file_path.is_file():
            print(" -", file_path.relative_to(local_model_dir))

    # Upload the complete directory into the Run's Artifacts.
    mlflow.log_artifacts(
        str(local_model_dir),
        artifact_path="xgboost_model_files"
    )

    feature_importance_df = pd.DataFrame({
        "Feature": X_train.columns.astype(str),
        "Importance": xgb_model.feature_importances_
    }).sort_values(
        by="Importance",
        ascending=False
    )

    plt.figure(figsize=(10, 6))

    sns.barplot(
        data=feature_importance_df.head(20),
        x="Importance",
        y="Feature"
    )

    plt.title("XGBoost Feature Importance")
    plt.tight_layout()

    plot_path = OUTPUT_DIR / f"xgb_feature_importance_v{version}.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close()

    mlflow.log_artifact(
        str(plot_path),
        artifact_path="plots"
    )

    # Log a text classification report
    report_path = OUTPUT_DIR / f"classification_report_v{version}.txt"

    with open(report_path, "w", encoding="utf-8") as f:
        f.write(
            classification_report(
                y_test,
                xgb_pred_mnist,
                zero_division=0
            )
        )

    mlflow.log_artifact(
        str(report_path),
        artifact_path="evaluation"
    )

    duration = time.time() - start_time
    mlflow.log_metric("training_and_logging_seconds", duration)

    print(f"Run completed in {duration:.2f} seconds.")
    print("Run ID:", run.info.run_id)
    print("Registered model:", REGISTERED_MODEL_NAME)
    print("Model URI:", model_info.model_uri)

print("Finished successfully.")

MLflow version: 3.17.2
Tracking URI: http://localhost:5000
Training rows: 48000
Validation rows: 12000
Number of features: 784
Run ID: 94c2aab469624ec4994b30ff1050eb6e
Experiment version: 4


c:\Users\USER\OneDrive\Desktop\Aya\venv_AI_Aya_ALL\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
c:\Users\USER\OneDrive\Desktop\Aya\venv_AI_Aya_ALL\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers Wit

Validation accuracy: 97.73%


Registered model 'MNIST_CLASSIFICATION_Predictor' already exists. Creating a new version of this model...
2026/10/26 22:50:06 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: MNIST_CLASSIFICATION_Predictor, version 2
Created version '2' of model 'MNIST_CLASSIFICATION_Predictor'.


Logged model URI: models:/m-9e5269d347d54485988a2c6dd58d8f8c
Logged model ID: m-9e5269d347d54485988a2c6dd58d8f8c
Local MLflow model files:
 - conda.yaml
 - input_example.json
 - MLmodel
 - model.ubj
 - python_env.yaml
 - requirements.txt
 - serving_input_example.json
Run completed in 262.53 seconds.
Run ID: 94c2aab469624ec4994b30ff1050eb6e
Registered model: MNIST_CLASSIFICATION_Predictor
Model URI: models:/m-9e5269d347d54485988a2c6dd58d8f8c
🏃 View run xgb_Mnist_Prediction_Experiments_v4_2026-10-26_22-45-44 at: http://localhost:5000/#/experiments/1/runs/94c2aab469624ec4994b30ff1050eb6e
🧪 View experiment at: http://localhost:5000/#/experiments/1
Finished successfully.
